In [1]:
!pip install transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [4]:
!pip install sentence-transformers PyPDF2 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.8 MB/s eta 0:00:00:00:0100:01


In [5]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss

In [6]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

In [7]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [8]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [9]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [10]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [11]:
pdf_path = "/kaggle/input/datasets/andrewsameh20/tips-hindawi-university-info/Tips Hindawi University Info.pdf"  

text = extract_text_from_pdf(pdf_path)
chunks = chunk_text(text,chunk_size=50, overlap=5)

model_embeddings, embeddings = embed_chunks(chunks)


index = create_faiss_index(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
question = "Where is Tips Hindawi University located?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more

--- Chunk 2 ---
leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research,

--- Chunk 3 ---
Four residence halls - A medical center - Sports complex and stadium - Innovation and entrepreneurship hub 2.2 Satellite Campuses North Campus : Focused on agricultural sciences and environmental research West Campus : Home to the School of Design and Architecture 2.3 St

In [13]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [14]:
answer = generate_text(prompt, max_length=700)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2479: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


In [15]:
print(answer[0])

Answer the next question: Where is Tips Hindawi University located? by reading the following text:1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 alumni who have made significant contributions to their communities and professions worldwide.

2. Campus and Location
   Tips Hindawi University is situated in the vibrant city of Baghdad, the capital of Iraq. The main campus spans over 500 acres, providing a lush green oasis amidst the bustling metropolis. The campus is strategically located near major transportation networks, making it easily accessible from all parts of the city and the country.

Based on the provided text, Tips Hindawi University is located in Baghdad, Iraq.


In [16]:
question = "Does the university oer online programs?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research,

--- Chunk 2 ---
of Students : Mr . Samer Hussein Registrar's Office : Handles academic records, course registration, transcripts Office of International Programs : Assists international students and partnerships 6. Research and Innovation Center for Renewable Energy Research Artificial Intelligence and Robotics Lab Public Policy and Development Institute Annual "Future Frontiers" Conference Research

--- Chunk 3 ---
Graduate Programs MBA (Master of Business Administration) MSc in Data Science MA in Psychology PhD in Mechanical Engineering PhD in Political Science 4. Admissions and Tuition 4.1 Undergraduate Admissions Hi

In [17]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [18]:
answer = generate_text(prompt, max_length=700)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [19]:
print(answer[0])

Answer the next question: Does the university oer online programs? by reading the following text:leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, innovation, and entrepreneurship. Additionally, THU offers a wide range of programs, including undergraduate and graduate degrees, online courses, and executive education programs. The university is committed to providing quality education and fostering lifelong learning for its students.

Yes, the university offers online programs.


In [20]:
question = "Is there financial aid for international students?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
interview for certain programs 4.2 Graduate Admissions Relevant bachelor’s degree Minimum GPA of 3.0/4.0 Two academic references Statement of purpose 4.3 Tuition Fees (Annual) Undergraduate: $5,000 - $8,000 Graduate: $6,500 - $10,000 4.4 Scholarships Merit Scholarships : 25% to 100% tuition coverage Need-Based Grants Research Fellowships for graduate students 5.

--- Chunk 2 ---
Graduate Programs MBA (Master of Business Administration) MSc in Data Science MA in Psychology PhD in Mechanical Engineering PhD in Political Science 4. Admissions and Tuition 4.1 Undergraduate Admissions High school GPA of 85% or equivalent English proficiency test (IELTS 6.0 or TOEFL 80) Entrance interview for certain programs 4.2

--- Chunk 3 ---
Fellowships for graduate students 5. Administration President : Dr . Nabil Al-Khatib• • • • • • • • • • • • • • • • • • • • • • • 2 Vice President of Academic Affairs : Prof. Layla Mahmoud Dean of Students : Mr .


In [21]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [22]:
answer = generate_text(prompt, max_length=700)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [23]:
print(answer[0])

Answer the next question: Is there financial aid for international students? by reading the following text:interview for certain programs 4.2 Graduate Admissions Relevant bachelor’s degree Minimum GPA of 3.0/4.0 Two academic references Statement of purpose 4.3 Tuition Fees (Annual) Undergraduate: $5,000 - $8,000 Graduate: $6,500 - $10,000 4.4 Scholarships Merit Scholarships : 25% to 100% tuition coverage Need-Based Grants Research Fellowships for graduate students 5. Financial Aid Available for International Students 5.1 Loans International students are not eligible for federal loans. Private loans are an option, but they usually have higher interest rates. 5.2 Scholarships Some universities and organizations offer scholarships specifically for international students. 5.3 Work-Study International students are eligible for on-campus work-study programs, but they must maintain their full-time student status.

The text states that international students are eligible for on-campus work-stu

In [24]:
question = "What languages are used for instruction?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
Events and Traditions Founders' Week Annual Cultural Festival Spring Research Showcase 7.3 Support Services Mental Health Counseling Career Development Center Tutoring and Writing Labs Disability Services 8. Faculty Highlights Dr. Yasir Al-Sabbagh (Computer Science): Expert in natural language processing Prof. Rana Khalil (Economics): World Bank consultant and author Dr. Omar

--- Chunk 2 ---
Dormitory (First-year undergraduates) Rafidain Apartments (Graduate students) Amal Housing Complex (Family and international students) 3. Academic Structure 3.1 Faculties Faculty of Engineering Faculty of Medicine and Health Sciences Faculty of Business and Economics Faculty of Arts and Humanities Faculty of Law and International Studies Faculty of Computer and Information Sciences

--- Chunk 3 ---
of Computer and Information Sciences Faculty of Architecture and Design• • • • • • • • • • • • 1 3.2 Sample Undergraduate Programs BSc in Computer Science BA in International Relations 

In [25]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [26]:
answer = generate_text(prompt, max_length=700)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [27]:
print(answer[0])

Answer the next question: What languages are used for instruction? by reading the following text:Events and Traditions Founders' Week Annual Cultural Festival Spring Research Showcase 7.3 Support Services Mental Health Counseling Career Development Center Tutoring and Writing Labs Disability Services 8. Faculty Highlights Dr. Yasir Al-Sabbagh (Computer Science): Expert in natural language processing Prof. Rana Khalil (Economics): World Bank consultant and author Dr. Omar Ghannam (Education): Pioneer in inclusive education Dr. Samar Helou (Psychology): Renowned researcher in cross-cultural psychology

The languages used for instruction at the university are English and Arabic.
